# Experiment 3: Generation Quality for 7 Missing Compressors

**Goal:** Fill the `---` entries in Table 2 (full 14-compressor table) with TA and ppl data.

**Missing compressors:** fp8_e4m3, kvquant_2bit, lossless_zstd, lossless_lz4, topk_prune_50, palu_int4, mixed_k8v4

**Models tested:** Llama-3.1-8B (robust) + Qwen2.5-3B (sensitive) — covers the two extremes.

- 8 compressors (7 missing + identity baseline) × 2 models × 50 prompts = 800 results
- **Runtime:** ~3-4 hours per model on A100, ~5-6 hours total
- **GPU requirement:** A100 recommended (7B model in FP16 needs ~16GB)

## Setup
1. Create zip locally: `cd KVShuttle && zip -r kvshuttle.zip . -x '.git/*' -x '*.pyc' -x '__pycache__/*'`
2. Set runtime to A100 GPU
3. Run all cells — upload zip when prompted
4. Paste your HF token for Llama access

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name}")
    print(f"Memory: {gpu_mem:.1f} GB")
    if gpu_mem < 30:
        print("WARNING: A100 (40GB) recommended for Llama-3.1-8B in FP16.")
        print("         T4 (15GB) may work for Qwen2.5-3B only.")
else:
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > GPU")

In [ ]:
# Install dependencies + upload KVShuttle
!pip install -q transformers accelerate datasets pyyaml tqdm

import os
if not os.path.exists("kvshuttle"):
    from google.colab import files
    print("Upload kvshuttle.zip")
    uploaded = files.upload()
    !unzip -qo kvshuttle.zip -d KVShuttle
    %cd KVShuttle
    !pip install -q -e .
else:
    print("KVShuttle already installed")

In [ ]:
# HuggingFace login (required for Llama-3.1-8B)
from huggingface_hub import login
login(token="PASTE_YOUR_HF_TOKEN_HERE")

In [ ]:
# Verify installation — check that all 7 missing compressors are available
from kvshuttle.models.loader_torch import TORCH_MODEL_REGISTRY
from kvshuttle.compression.registry import list_compressors

available = list_compressors()
required = ['fp8_e4m3', 'kvquant_2bit', 'lossless_zstd', 'lossless_lz4',
            'topk_prune_50', 'palu_int4', 'mixed_k8v4', 'identity']
missing = [c for c in required if c not in available]
print(f"Available compressors: {available}")
if missing:
    print(f"WARNING: Missing compressors: {missing}")
else:
    print("All required compressors available!")

In [ ]:
# Write config and run experiment
import yaml
from pathlib import Path

# Choose models based on GPU memory
import torch
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9

if gpu_mem >= 30:  # A100
    models = ["llama-3.1-8b", "qwen2.5-3b"]
    print("A100 detected: running both Llama-3.1-8B and Qwen2.5-3B")
else:  # T4
    models = ["qwen2.5-3b"]
    print("T4 detected: running Qwen2.5-3B only (re-run on A100 for Llama-3.1-8B)")

config = {
    "experiment": {
        "name": "generation_quality_fp16_missing7",
        "description": "Generation quality for 7 missing compressors"
    },
    "backend": "torch",
    "models": models,
    "compressors": [
        "identity",
        "fp8_e4m3",
        "kvquant_2bit",
        "lossless_zstd",
        "lossless_lz4",
        "topk_prune_50",
        "palu_int4",
        "mixed_k8v4"
    ],
    "bandwidths_gbps": [10],
    "prompts": {
        "source": "wikitext",
        "count": 50,
        "min_tokens": 128,
        "max_tokens": 512
    },
    "evaluation": {
        "attention_error": True,
        "perplexity": True,
        "token_agreement": True
    },
    "output": {
        "dir": "experiments/results/generation_quality_fp16_missing7",
        "save_per_layer": False
    }
}

config_path = Path("experiments/configs/generation_quality_torch_missing7.yaml")
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, "w") as f:
    yaml.dump(config, f)

n_results = len(models) * len(config['compressors']) * config['prompts']['count']
print(f"Config: {config_path}")
print(f"Models: {models}")
print(f"Compressors: {config['compressors']}")
print(f"Expected results: {n_results}")

!python -m experiments.scripts.run_experiment {config_path}

In [ ]:
# Inspect results
import json
import pandas as pd
from pathlib import Path

results_path = Path("experiments/results/generation_quality_fp16_missing7/results.json")

if results_path.exists():
    with open(results_path) as f:
        data = json.load(f)
    print(f"Results: {len(data['results'])} entries")
    print(f"Models: {data['metadata'].get('models', [])}")

    df = pd.DataFrame(data['results'])
    cols = ['mean_key_cosine_sim', 'token_agreement', 'perplexity_delta', 'compression_ratio']
    agg_cols = {c: 'mean' for c in cols if c in df.columns}
    if agg_cols:
        summary = df.groupby(['model', 'compressor']).agg(agg_cols).round(4)
        display(summary)
        print("\n=== Key findings ===")
        for model in df['model'].unique():
            mdf = df[df['model'] == model]
            for comp in sorted(mdf['compressor'].unique()):
                if comp == 'identity':
                    continue
                cdf = mdf[mdf['compressor'] == comp]
                ta = cdf['token_agreement'].mean() if 'token_agreement' in cdf else 'N/A'
                print(f"  {model} + {comp}: TA={ta:.3f}" if isinstance(ta, float) else f"  {model} + {comp}: TA={ta}")
else:
    print("Results not found. Check experiment output above.")

In [ ]:
# Download results
if results_path.exists():
    from google.colab import files
    files.download(str(results_path))
    print(f"Downloaded {results_path}")
    print("\nNext steps:")
    print("1. Copy to experiments/results/generation_quality_fp16_missing7/")
    print("2. Run: python experiments/scripts/merge_fp16_results_v2.py")
    print("3. Run: python experiments/scripts/compute_confidence_intervals.py")
    print("4. Run: python experiments/scripts/generate_paper_tables.py --full")